# Feature Selection — Pre-Snap Situational Variables

**Research context:** Binary classification of NFL play-calls (pass vs. run) using only pre-snap situational variables.

Feature selection proceeds in **three stages**:

| Stage | Method | Purpose |
|---|---|---|
| 1 | Structural exclusion (assisted by Claude Sonnet 4.6, verified by author) | Remove post-snap outcomes, player identities, administrative columns, and play-type derivates |
| 2 | Missing-value filter | Remove columns with >40% missing values |
| 3 | Statistical association | Point-biserial *r* (continuous) and Cramér's *V* (categorical) against binary target |

> This approach ensures feature inclusion is **data-driven and scientifically documentable**, with each exclusion explicitly justified.

## 0 — Setup

In [1]:
import sys
sys.path.append('..') #to find local models as src or config

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from matplotlib.patches import Patch

from src.data_loader import load_data
from config import FIGURES_EDA, PLOT_DPI, PLOT_PALETTE

FIGURES_EDA.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette=PLOT_PALETTE)
print("Setup complete.")

Setup complete.


## 1 — Load & Filter to Pass/Run Plays

In [2]:
df_raw = load_data()
df = df_raw[df_raw["play_type"].isin(["pass", "run"])].copy()

# join every 15 col with a comma separtor to get better view
cols = df.columns.tolist()
formatted_cols = "\n".join([", ".join(cols[i:i+15]) for i in range(0, len(cols), 15)])

print("Columns:\n" + formatted_cols)

Cache found - loading data from: C:\Users\rapha\.kiro\projects\Predicting-American-Football-Plays\data\cache\pbp_raw.parquet
Columns:
play_id, game_id, old_game_id, home_team, away_team, season_type, week, posteam, posteam_type, defteam, side_of_field, yardline_100, game_date, quarter_seconds_remaining, half_seconds_remaining
game_seconds_remaining, game_half, quarter_end, drive, sp, qtr, down, goal_to_go, time, yrdln, ydstogo, ydsnet, desc, play_type, yards_gained
shotgun, no_huddle, qb_dropback, qb_kneel, qb_spike, qb_scramble, pass_length, pass_location, air_yards, yards_after_catch, run_location, run_gap, field_goal_result, kick_distance, extra_point_result
two_point_conv_result, home_timeouts_remaining, away_timeouts_remaining, timeout, timeout_team, td_team, td_player_name, td_player_id, posteam_timeouts_remaining, defteam_timeouts_remaining, total_home_score, total_away_score, posteam_score, defteam_score, score_differential
posteam_score_post, defteam_score_post, score_differen

## Stage 1 — Structural Column Exclusion

> **Methodological note:** The nflverse play-by-play dataset contains approximately 400 columns. A full manual review of each column was conducted with the assistance of **Claude Sonnet 4.6 (Anthropic, 2025)**. All exclusions were subsequently verified by the author against the [nflverse data dictionary](https://nflreadr.nflverse.com/articles/dictionary_pbp.html). Each column was assessed against two criteria: (1) is it observable *before* the snap, and (2) does it represent situational game context rather than team/player identity?

Columns are excluded under **six categories**:

---

### 1.1 — Post-Snap Outcomes
These columns describe what happened *after* the ball was snapped. Including them would constitute **data leakage** — the model would have access to information that cannot exist at prediction time.

| Column(s) | Reason |
|---|---|
| `yards_gained`, `air_yards`, `yards_after_catch`, `ydsnet`  | Yardage realized after the play |
| `passing_yards`, `rushing_yards`, `receiving_yards` | Play-level outcome yardage |
| `posteam_score_post`, `defteam_score_post`, `score_differential_post` | Score *after* the play |
| `first_down`, `series_result`, `series_success`, `fixed_drive_result` | Whether a first down / drive succeeded |
| `end_clock_time`, `end_yard_line` | State at end of play |
| `result` | Final game result |
| `aborted_play`, `play_deleted` | Post-snap administrative flags |
| `return_yards`, `return_team` | Post-snap return outcome |
| `lateral_reception`, `lateral_rush`, `lateral_return`, `lateral_recovery`, `lateral_receiving_yards`, `lateral_rushing_yards` | Post-snap lateral ball movement |

---

### 1.2 — EPA / WPA Metrics
Expected Points Added and Win Probability Added are **model-derived post-snap outcomes**. `ep` and `wp` (pre-snap versions) are retained as candidates.

| Column(s) | Reason |
|---|---|
| `epa`, `air_epa`, `yac_epa`, `comp_air_epa`, `comp_yac_epa`, `qb_epa` | EPA realized post-snap |
| `total_home_epa`, `total_away_epa`, `total_home_rush_epa`, `total_away_rush_epa`, `total_home_pass_epa`, `total_away_pass_epa` | Cumulative EPA — post-snap |
| `total_home_raw_air_epa`, `total_away_raw_air_epa`, `total_home_raw_yac_epa`, `total_away_raw_yac_epa` | Cumulative raw EPA — post-snap |
| `total_home_comp_air_epa`, `total_away_comp_air_epa`, `total_home_comp_yac_epa`, `total_away_comp_yac_epa` | Cumulative comp EPA — post-snap |
| `wpa`, `vegas_wpa`, `vegas_home_wpa`, `air_wpa`, `yac_wpa`, `comp_air_wpa`, `comp_yac_wpa` | WPA — realized post-snap |
| `total_home_rush_wpa`, `total_away_rush_wpa`, `total_home_pass_wpa`, `total_away_pass_wpa` | Cumulative WPA — post-snap |
| `total_home_raw_air_wpa`, `total_away_raw_air_wpa`, `total_home_raw_yac_wpa`, `total_away_raw_yac_wpa` | Cumulative raw WPA — post-snap |
| `total_home_comp_air_wpa`, `total_away_comp_air_wpa`, `total_home_comp_yac_wpa`, `total_away_comp_yac_wpa` | Cumulative comp WPA — post-snap |
| `home_wp_post`, `away_wp_post` | Win probability *after* the play |
| `xyac_epa`, `xyac_mean_yardage`, `xyac_median_yardage`, `xyac_success`, `xyac_fd` | Expected YAC metrics — post-snap model outputs |
| `cp`, `cpoe` | Completion probability over expected — post-snap |
| `xpass`, `pass_oe` | Expected pass rate / pass over expected — circular leakage (derived from the same target) |

---

### 1.3 — Play-Type Derivates (Target Leakage)
These columns are **directly derived from or equivalent to** `play_type`. Including them would trivially solve the classification task.

| Column(s) | Reason |
|---|---|
| `pass_attempt`, `rush_attempt` | Binary flags identical to the target |
| `pass`, `rush` | Equivalent binary indicators |
| `play_type_nfl`, `play` | Alternative play type encodings |
| `first_down_pass`, `first_down_rush` | Outcome conditioned on play type |
| `pass_touchdown`, `rush_touchdown`, `return_touchdown`, `touchdown` | Outcome conditioned on play type |
| `complete_pass`, `incomplete_pass`, `interception`, `sack` | Pass-specific post-snap outcomes |
| `pass_length`, `pass_location`, `run_location`, `run_gap` | Post-snap play-call details |
| `qb_dropback`, `qb_scramble`, `qb_kneel`, `qb_spike` | QB actions that reveal or are the play type |
| `third_down_converted`, `third_down_failed`, `fourth_down_converted`, `fourth_down_failed` | Post-snap conversion outcomes |
| `first_down_penalty`, `penalty`, `penalty_team`, `penalty_type`, `penalty_yards` | Post-snap penalty events |
| `fumble`, `fumble_lost`, `fumble_forced`, `fumble_not_forced`, `fumble_out_of_bounds` | Post-snap fumble events |
| `fumble_recovery_1_team`, `fumble_recovery_1_yards`, `fumble_recovery_2_team`, `fumble_recovery_2_yards` | Post-snap fumble recovery |
| `fumbled_1_team`, `fumbled_2_team` | Post-snap fumble events |
| `tackled_for_loss`, `qb_hit`, `safety`, `touchback`, `out_of_bounds` | Post-snap play events |
| `success` | Binary outcome metric — post-snap |

---

### 1.4 — Player & Team Identity
Player and team identifiers are **not situational context**. The model should learn game-state patterns, not team or player tendencies.

| Column(s) | Reason |
|---|---|
| `passer_player_id`, `passer_player_name`, `passer_id`, `passer`, `passer_jersey_number` | Passer identity |
| `rusher_player_id`, `rusher_player_name`, `rusher_id`, `rusher`, `rusher_jersey_number` | Rusher identity |
| `receiver_player_id`, `receiver_player_name`, `receiver_id`, `receiver`, `receiver_jersey_number` | Receiver identity |
| `td_player_id`, `td_player_name`, `td_team` | Scoring player identity |
| `home_coach`, `away_coach` | Coaching identity — not situational |
| `name`, `id`, `jersey_number`, `fantasy_player_name`, `fantasy_player_id`, `fantasy`, `fantasy_id` | Generic player identity columns |
| `lateral_receiver_player_id/name`, `lateral_rusher_player_id/name`, `lateral_sack_player_id/name` | Lateral play player IDs |
| `lateral_interception_player_id/name`, `punt_returner_player_id/name` | Special teams player IDs |
| `kickoff_returner_player_id/name`, `lateral_kickoff_returner_player_id/name` | Special teams player IDs |
| `punter_player_id/name`, `kicker_player_name/id` | Special teams player IDs |
| `interception_player_id/name`, `lateral_interception_player_id/name` | Post-snap player identity |
| `own_kickoff_recovery_player_id/name`, `blocked_player_id/name` | Post-snap player identity |
| `tackle_for_loss_1/2_player_id/name`, `qb_hit_1/2_player_id/name` | Post-snap player identity |
| `forced_fumble_player_1/2_team/player_id/name` | Post-snap player identity |
| `solo_tackle_1/2_team/player_id/name`, `assist_tackle_1/2/3/4_player_id/name/team` | Post-snap tackle identity |

---

### 1.5 — Special Teams (Irrelevant after play_type filter)
All special teams columns are structurally irrelevant after filtering to `play_type ∈ {pass, run}`.

| Column(s) | Reason |
|---|---|
| `punt_blocked`, `punt_inside_twenty`, `punt_in_endzone`, `punt_out_of_bounds`, `punt_downed`, `punt_fair_catch` | Punt events |
| `kickoff_inside_twenty`, `kickoff_in_endzone`, `kickoff_out_of_bounds`, `kickoff_downed`, `kickoff_fair_catch` | Kickoff events |
| `field_goal_result`, `kick_distance`, `extra_point_result`, `two_point_conv_result` | Kicking outcomes |
| `extra_point_attempt`, `two_point_attempt`, `field_goal_attempt`, `kickoff_attempt`, `punt_attempt` | Special teams attempt flags |
| `own_kickoff_recovery`, `own_kickoff_recovery_td` | Kickoff recovery events |
| `defensive_extra_point_attempt`, `defensive_extra_point_conv`, `defensive_two_point_attempt`, `defensive_two_point_conv` | Defensive special teams |
| `special`, `special_teams_play`, `st_play_type` | Special teams play flags |

---

### 1.6 — Administrative / No Predictive Content

| Column(s) | Reason |
|---|---|
| `play_id`, `game_id`, `old_game_id`, `nfl_api_id`, `stadium_id` | Row/game identifiers |
| `desc`, `time`, `yrdln` | Text descriptions / redundant time/field representations |
| `game_date`, `start_time`, `time_of_day`, `drive_real_start_time` | Timestamps |
| `game_half`, `quarter_end`, `order_sequence`, `play_clock`, `qtr`, `half_seconds_remaining` | Administrative sequencing |
| `drive`, `sp`, `fixed_drive`, `series` | Drive/series counter IDs |
| `drive_play_id_started`, `drive_play_id_ended` | Drive boundary IDs |
| `drive_game_clock_start`, `drive_game_clock_end` | Drive clock boundaries |
| `drive_quarter_start`, `drive_quarter_end` | Drive quarter boundaries |
| `drive_start_transition`, `drive_end_transition` | How the drive started/ended |
| `drive_start_yard_line`, `drive_end_yard_line` | Drive start/end field position |
| `drive_inside20`, `drive_ended_with_score`, `drive_first_downs`, `drive_play_count`, `drive_yards_penalized`, `drive_time_of_possession` | Cumulative drive stats — only known after drive ends |
| `game_stadium`, `stadium`, `weather` | Redundant with `roof`, `surface`, `location` |
| `home_team`, `away_team`, `posteam_type`, `side_of_field`, `season_type` | Team identity / redundant field info |
| `td_team`, `timeout_team`, `timeout` | Post-snap event team identity |
| `replay_or_challenge`, `replay_or_challenge_result` | Post-snap officiating |
| `home_opening_kickoff` | Pre-game coin toss — not situational |
| `away_timeouts_remaining`, `home_timeouts_remaining` | Redundant with `posteam_timeouts_remaining` / `defteam_timeouts_remaining` (situationally correct versions) |
| `away_score`, `home_score`, `total` | Redundant with `score_differential`, `posteam_score`, `defteam_score` |
| `away_wp`, `home_wp` | Redundant with `wp` / `def_wp` (possession-relative versions preferred) |
| `no_score_prob`, `opp_fg_prob`, `opp_safety_prob`, `opp_td_prob`, `fg_prob`, `safety_prob`, `td_prob`, `extra_point_prob`, `two_point_conversion_prob` | Pre-snap model probabilities — highly correlated with `wp`/`ep`; introduce model-in-model dependency |
| `play_type` | The target variable itself — must be excluded to prevent leakage |
| `posteam`, `defteam` | Team identity — not situational context |
| `total_home_score`, `total_away_score`, `posteam_score`, `defteam_score`  | Redundant with `posteam_score`, `defteam_score`, and `score_differential` |
| `def_wp`, `vegas_wp`, `vegas_home_wp` | Redundant with `wp` (possession-relative); `def_wp = 1 − wp` by definition |

---

### 1.7 — Post-Snap Player Identities (missed in initial pass)
The following player identity columns were not caught by earlier rules and are excluded here.

| Column(s) | Reason |
|---|---|
| `fumble_recovery_1/2_player_id/name`, `fumbled_1/2_player_id/name` | Post-snap fumble player identity |
| `half_sack_1/2_player_id/name`, `sack_player_id/name` | Post-snap sack player identity |
| `pass_defense_1/2_player_id/name` | Post-snap defensive player identity |
| `penalty_player_id/name`, `safety_player_id/name` | Post-snap event player identity |
| `tackle_with_assist_1/2_player_id/name` | Post-snap tackle player identity |

In [3]:
# Stage 1 — hardcoded exclusion list (reviewed and verified by author)
# Generated with assistance of Claude Sonnet 4.6 (Anthropic, 2025)

EXCLUDED_STAGE1 = [
    # 1.1 Post-snap outcomes
    "yards_gained", "air_yards", "yards_after_catch",
    "passing_yards", "rushing_yards", "receiving_yards",
    "posteam_score_post", "defteam_score_post", "score_differential_post",
    "first_down", "series_result", "series_success", "fixed_drive_result",
    "end_clock_time", "end_yard_line", "result",
    "aborted_play", "play_deleted",
    "return_yards", "return_team",
    "lateral_reception", "lateral_rush", "lateral_return", "lateral_recovery",
    "lateral_receiving_yards", "lateral_rushing_yards",
    "ydsnet",
    # 1.2 EPA / WPA
    "epa", "ep", "air_epa", "yac_epa", "comp_air_epa", "comp_yac_epa", "qb_epa",
    "total_home_epa", "total_away_epa",
    "total_home_rush_epa", "total_away_rush_epa",
    "total_home_pass_epa", "total_away_pass_epa",
    "total_home_raw_air_epa", "total_away_raw_air_epa",
    "total_home_raw_yac_epa", "total_away_raw_yac_epa",
    "total_home_comp_air_epa", "total_away_comp_air_epa",
    "total_home_comp_yac_epa", "total_away_comp_yac_epa",
    "wpa", "vegas_wpa", "vegas_home_wpa",
    "air_wpa", "yac_wpa", "comp_air_wpa", "comp_yac_wpa",
    "total_home_rush_wpa", "total_away_rush_wpa",
    "total_home_pass_wpa", "total_away_pass_wpa",
    "total_home_raw_air_wpa", "total_away_raw_air_wpa",
    "total_home_raw_yac_wpa", "total_away_raw_yac_wpa",
    "total_home_comp_air_wpa", "total_away_comp_air_wpa",
    "total_home_comp_yac_wpa", "total_away_comp_yac_wpa",
    "home_wp_post", "away_wp_post",
    "xyac_epa", "xyac_mean_yardage", "xyac_median_yardage", "xyac_success", "xyac_fd",
    "cp", "cpoe", "xpass", "pass_oe",
    # 1.3 Play-type derivates
    "pass_attempt", "rush_attempt", "pass", "rush",
    "play_type_nfl", "play",
    "first_down_pass", "first_down_rush",
    "pass_touchdown", "rush_touchdown", "return_touchdown", "touchdown",
    "complete_pass", "incomplete_pass", "interception", "sack",
    "pass_length", "pass_location", "run_location", "run_gap",
    "qb_dropback", "qb_scramble", "qb_kneel", "qb_spike",
    "third_down_converted", "third_down_failed",
    "fourth_down_converted", "fourth_down_failed",
    "first_down_penalty", "penalty", "penalty_team", "penalty_type", "penalty_yards",
    "fumble", "fumble_lost", "fumble_forced", "fumble_not_forced", "fumble_out_of_bounds",
    "fumble_recovery_1_team", "fumble_recovery_1_yards",
    "fumble_recovery_2_team", "fumble_recovery_2_yards",
    "fumbled_1_team", "fumbled_2_team",
    "tackled_for_loss", "qb_hit", "safety", "touchback", "out_of_bounds",
    "success",
    # 1.4 Player & team identity
    "passer_player_id", "passer_player_name", "passer_id", "passer", "passer_jersey_number",
    "rusher_player_id", "rusher_player_name", "rusher_id", "rusher", "rusher_jersey_number",
    "receiver_player_id", "receiver_player_name", "receiver_id", "receiver", "receiver_jersey_number",
    "td_player_id", "td_player_name", "td_team",
    "home_coach", "away_coach",
    "name", "id", "jersey_number",
    "fantasy_player_name", "fantasy_player_id", "fantasy", "fantasy_id",
    "lateral_receiver_player_id", "lateral_receiver_player_name",
    "lateral_rusher_player_id", "lateral_rusher_player_name",
    "lateral_sack_player_id", "lateral_sack_player_name",
    "lateral_interception_player_id", "lateral_interception_player_name",
    "punt_returner_player_id", "punt_returner_player_name",
    "lateral_punt_returner_player_id", "lateral_punt_returner_player_name",
    "kickoff_returner_player_id", "kickoff_returner_player_name",
    "lateral_kickoff_returner_player_id", "lateral_kickoff_returner_player_name",
    "punter_player_id", "punter_player_name",
    "kicker_player_name", "kicker_player_id",
    "interception_player_id", "interception_player_name",
    "own_kickoff_recovery_player_id", "own_kickoff_recovery_player_name",
    "blocked_player_id", "blocked_player_name",
    "tackle_for_loss_1_player_id", "tackle_for_loss_1_player_name",
    "tackle_for_loss_2_player_id", "tackle_for_loss_2_player_name",
    "qb_hit_1_player_id", "qb_hit_1_player_name",
    "qb_hit_2_player_id", "qb_hit_2_player_name",
    "forced_fumble_player_1_team", "forced_fumble_player_1_player_id", "forced_fumble_player_1_player_name",
    "forced_fumble_player_2_team", "forced_fumble_player_2_player_id", "forced_fumble_player_2_player_name",
    "solo_tackle_1_team", "solo_tackle_2_team",
    "solo_tackle_1_player_id", "solo_tackle_2_player_id",
    "solo_tackle_1_player_name", "solo_tackle_2_player_name",
    "assist_tackle_1_player_id", "assist_tackle_1_player_name", "assist_tackle_1_team",
    "assist_tackle_2_player_id", "assist_tackle_2_player_name", "assist_tackle_2_team",
    "assist_tackle_3_player_id", "assist_tackle_3_player_name", "assist_tackle_3_team",
    "assist_tackle_4_player_id", "assist_tackle_4_player_name", "assist_tackle_4_team",
    # 1.5 Special teams
    "punt_blocked", "punt_inside_twenty", "punt_in_endzone", "punt_out_of_bounds",
    "punt_downed", "punt_fair_catch",
    "kickoff_inside_twenty", "kickoff_in_endzone", "kickoff_out_of_bounds",
    "kickoff_downed", "kickoff_fair_catch",
    "field_goal_result", "kick_distance", "extra_point_result", "two_point_conv_result",
    "extra_point_attempt", "two_point_attempt", "field_goal_attempt",
    "kickoff_attempt", "punt_attempt",
    "own_kickoff_recovery", "own_kickoff_recovery_td",
    "defensive_extra_point_attempt", "defensive_extra_point_conv",
    "defensive_two_point_attempt", "defensive_two_point_conv",
    "special", "special_teams_play", "st_play_type",
    # 1.6 Administrative
    "play_id", "game_id", "old_game_id", "nfl_api_id", "stadium_id",
    "desc", "time", "yrdln",
    "game_date", "start_time", "time_of_day", "drive_real_start_time",
    "game_half", "quarter_end", "order_sequence", "play_clock",
    "qtr", "half_seconds_remaining",
    "drive", "sp", "fixed_drive", "series",
    "drive_play_id_started", "drive_play_id_ended",
    "drive_game_clock_start", "drive_game_clock_end",
    "drive_quarter_start", "drive_quarter_end",
    "drive_start_transition", "drive_end_transition",
    "drive_start_yard_line", "drive_end_yard_line",
    "drive_inside20", "drive_ended_with_score", "drive_first_downs",
    "drive_play_count", "drive_yards_penalized", "drive_time_of_possession",
    "game_stadium", "stadium", "weather",
    "home_team", "away_team", "posteam_type", "side_of_field", "season_type",
    "td_team", "timeout_team", "timeout",
    "replay_or_challenge", "replay_or_challenge_result",
    "home_opening_kickoff",
    "away_timeouts_remaining", "home_timeouts_remaining",
    "away_score", "home_score", "total",
    "away_wp", "home_wp",
    "no_score_prob", "opp_fg_prob", "opp_safety_prob", "opp_td_prob",
    "fg_prob", "safety_prob", "td_prob", "extra_point_prob", "two_point_conversion_prob",
    # Tackle post-snap events not caught above
    "assist_tackle", "assist_tackle_1_team",
    "solo_tackle", "tackle_with_assist", "tackle_with_assist_1_team", "tackle_with_assist_2_team",
    # Post-snap player identities missed in first pass
    "fumble_recovery_1_player_id", "fumble_recovery_1_player_name",
    "fumble_recovery_2_player_id", "fumble_recovery_2_player_name",
    "fumbled_1_player_id", "fumbled_1_player_name",
    "fumbled_2_player_id", "fumbled_2_player_name",
    "half_sack_1_player_id", "half_sack_1_player_name",
    "half_sack_2_player_id", "half_sack_2_player_name",
    "sack_player_id", "sack_player_name",
    "pass_defense_1_player_id", "pass_defense_1_player_name",
    "pass_defense_2_player_id", "pass_defense_2_player_name",
    "penalty_player_id", "penalty_player_name",
    "safety_player_id", "safety_player_name",
    "tackle_with_assist_1_player_id", "tackle_with_assist_1_player_name",
    "tackle_with_assist_2_player_id", "tackle_with_assist_2_player_name",
    # Target itself and team identity
    "play_type", "posteam", "defteam","posteam_score", "defteam_score",
    # Redundant score columns (score_differential, posteam_score, defteam_score are sufficient)
    "total_home_score", "total_away_score",
    # Redundant WP columns (wp is possession-relative and sufficient)
    "def_wp", "vegas_wp", "vegas_home_wp",
]

# Only exclude columns that actually exist in the dataframe
EXCLUDED_STAGE1 = [c for c in EXCLUDED_STAGE1 if c in df.columns]

remaining_after_stage1 = sorted([c for c in df.columns if c not in EXCLUDED_STAGE1 and c != "target"])

print(f"Stage 1 — Columns excluded: {len(EXCLUDED_STAGE1)}")
print(f"Stage 1 — Columns remaining: {len(remaining_after_stage1)}")

# join every 5 col with a comma separtor to get better view
formatted_after_stage1= "\n".join([", ".join(remaining_after_stage1[i:i+5]) for i in range(0, len(remaining_after_stage1), 5)])

print("\nColumns:\n" + formatted_after_stage1)

Stage 1 — Columns excluded: 352
Stage 1 — Columns remaining: 22

Columns:
defteam_timeouts_remaining, div_game, down, game_seconds_remaining, goal_to_go
location, no_huddle, posteam_timeouts_remaining, quarter_seconds_remaining, roof
score_differential, season, shotgun, spread_line, surface
temp, total_line, week, wind, wp
yardline_100, ydstogo


## Stage 2 — Missing-Value Filter

Columns with more than **40% missing values** are excluded. A high missingness rate indicates the variable is not reliably recorded across game situations and would introduce systematic bias into the model.

In [4]:
MISSING_THRESHOLD = 0.40

missing_rate = df[remaining_after_stage1].isnull().mean().sort_values(ascending=False)

too_many_missing = missing_rate[missing_rate > MISSING_THRESHOLD].index.tolist()
remaining_after_stage2 = [c for c in remaining_after_stage1 if c not in too_many_missing]

print(f"Stage 2 — Columns excluded (>{MISSING_THRESHOLD:.0%} missing): {len(too_many_missing)}")
print(f"Stage 2 — Columns remaining:                                   {len(remaining_after_stage2)}")

if too_many_missing:
    print(f"\nExcluded:")
    for col in too_many_missing:
        print(f"  {col:<45} {missing_rate[col]:.1%} missing")

print(f"\nRemaining columns:\n{sorted(remaining_after_stage2)}")

Stage 2 — Columns excluded (>40% missing): 0
Stage 2 — Columns remaining:                                   22

Remaining columns:
['defteam_timeouts_remaining', 'div_game', 'down', 'game_seconds_remaining', 'goal_to_go', 'location', 'no_huddle', 'posteam_timeouts_remaining', 'quarter_seconds_remaining', 'roof', 'score_differential', 'season', 'shotgun', 'spread_line', 'surface', 'temp', 'total_line', 'week', 'wind', 'wp', 'yardline_100', 'ydstogo']


## Stage 3 — Statistical Association

For each remaining column, compute its association with the binary target (`1=pass`, `0=run`):

- **Continuous columns** → **Point-biserial correlation** $r_{pb}$  
  Mathematically identical to Pearson's $r$ when one variable is binary (Glass & Hopkins, 1996).  
  Range: $[-1, 1]$, signed (negative = more associated with run).

- **Categorical columns** → **Cramér's $V$**  
  Based on the $\chi^2$ statistic, normalized to $[0, 1]$.  
  $V = \sqrt{\chi^2 / (n \cdot \min(r-1, c-1))}$

In [5]:
def point_biserial(series, target):
    mask = series.notna() & target.notna()
    r, p = stats.pointbiserialr(series[mask], target[mask])
    return r, p, int(mask.sum())

def cramers_v(series, target):
    mask = series.notna() & target.notna()
    ct = pd.crosstab(series[mask].astype(str), target[mask])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    n = int(mask.sum())
    k = min(ct.shape) - 1
    v = float(np.sqrt(chi2 / (n * k))) if k > 0 else 0.0
    return v, p, n

results = []

for col in remaining_after_stage2:
    series = df[col]
    dtype = series.dtype
    n_unique = series.nunique(dropna=True)

    # Decide: categorical if object/bool or few unique integer values (<=10)
    is_categorical = dtype == object or dtype == bool or (dtype in ["int8", "int16", "int32", "int64"] and n_unique <= 10)

    try:
        if is_categorical:
            stat, p, n = cramers_v(series, df["target"])
            metric = "Cramér's V"
            signed_stat = stat
        else:
            stat, p, n = point_biserial(pd.to_numeric(series, errors="coerce"), df["target"])
            metric = "Point-biserial r"
            signed_stat = stat
            stat = abs(stat)

        results.append({
            "feature": col,
            "dtype": str(dtype),
            "n_unique": n_unique,
            "metric": metric,
            "signed_stat": signed_stat,
            "abs_stat": stat,
            "p_value": p,
            "n": n,
            "missing_pct": df[col].isnull().mean(),
        })
    except Exception as e:
        print(f"  SKIP {col}: {e}")

results_df = pd.DataFrame(results).sort_values("abs_stat", ascending=False).reset_index(drop=True)
results_df["significant"] = results_df["p_value"] < 0.001

print(f"Features computed: {len(results_df)}")
print(results_df[["feature", "metric", "signed_stat", "abs_stat", "p_value", "significant", "missing_pct"]].to_string(index=False))

  SKIP defteam_timeouts_remaining: 'target'
  SKIP div_game: 'target'
  SKIP down: 'target'
  SKIP game_seconds_remaining: 'target'
  SKIP goal_to_go: 'target'
  SKIP location: 'target'
  SKIP no_huddle: 'target'
  SKIP posteam_timeouts_remaining: 'target'
  SKIP quarter_seconds_remaining: 'target'
  SKIP roof: 'target'
  SKIP score_differential: 'target'
  SKIP season: 'target'
  SKIP shotgun: 'target'
  SKIP spread_line: 'target'
  SKIP surface: 'target'
  SKIP temp: 'target'
  SKIP total_line: 'target'
  SKIP week: 'target'
  SKIP wind: 'target'
  SKIP wp: 'target'
  SKIP yardline_100: 'target'
  SKIP ydstogo: 'target'


KeyError: 'abs_stat'

## Stage 3 — Visualisation: Full Ranked Association Plot

In [ ]:
plot_df = results_df.sort_values("abs_stat", ascending=True)

fig_height = max(6, len(plot_df) * 0.32)
fig, ax = plt.subplots(figsize=(10, fig_height))

colors = ["#2ecc71" if sig else "#e74c3c" for sig in plot_df["significant"]]

bars = ax.barh(
    plot_df["feature"],
    plot_df["abs_stat"],
    color=colors,
    edgecolor="white",
    linewidth=0.4,
)

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(
        bar.get_width() + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f"{row['abs_stat']:.3f}",
        va="center", fontsize=7, color="#333333"
    )

legend_elements = [
    Patch(facecolor="#2ecc71", label="p < 0.001 (significant)"),
    Patch(facecolor="#e74c3c", label="p ≥ 0.001 (not significant)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
ax.set_xlabel("Association Strength  |  Point-biserial |r| or Cramér's V", fontsize=9)
ax.set_title(
    "Stage 3: Statistical Association of Pre-Snap Features with Play Type\n"
    "(Pass = 1, Run = 0)  —  after structural and missing-value filtering",
    fontsize=10, pad=12
)
ax.set_xlim(0, plot_df["abs_stat"].max() + 0.12)
sns.despine(left=True)
plt.tight_layout()

out_path = FIGURES_EDA / "feature_association_all.png"
fig.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

## Stage 3 — Multicollinearity Check

Among the top candidate features, check for pairwise Pearson correlation to detect redundancy.  
A threshold of $|r| > 0.70$ indicates potential multicollinearity (relevant especially for Logistic Regression).

In [ ]:
# Use top N features by association strength for the collinearity check
TOP_N = 20
top_features = results_df.head(TOP_N)["feature"].tolist()

# Only numeric columns for Pearson
numeric_top = [c for c in top_features if pd.api.types.is_numeric_dtype(df[c])]

corr_matrix = df[numeric_top].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt=".2f",
    cmap="coolwarm", center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.7},
    annot_kws={"size": 7},
)
ax.set_title(
    f"Pearson Correlation — Top {TOP_N} Pre-Snap Feature Candidates\n"
    "Pairs with |r| > 0.70 indicate potential multicollinearity",
    fontsize=10
)
plt.tight_layout()

out_path = FIGURES_EDA / "feature_collinearity_top20.png"
fig.savefig(out_path, dpi=PLOT_DPI, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

# Print high-collinearity pairs
print("\nHigh collinearity pairs (|r| > 0.70):")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.70:
            c1 = corr_matrix.columns[i]
            c2 = corr_matrix.columns[j]
            print(f"  {c1:<40} ↔  {c2:<40} r = {r:.3f}")

## Final Feature List

Based on the three stages above, define the final feature set.  
**Fill this in after reviewing the charts.**  
For each excluded high-correlation pair, keep the feature with the higher association strength.

In [ ]:
# TODO: fill in after inspecting charts
# Start from top features; remove one of each highly collinear pair
SELECTED_FEATURES = [
    # Replace this list after running the notebook
]

if SELECTED_FEATURES:
    print("Final feature set:")
    print(f"{'Feature':<40} {'Metric':<20} {'|Stat|':>7}  {'p-value':>10}")
    print("-" * 82)
    for f in SELECTED_FEATURES:
        row = results_df[results_df["feature"] == f]
        if not row.empty:
            r = row.iloc[0]
            print(f"{f:<40} {r['metric']:<20} {r['abs_stat']:>7.4f}  {r['p_value']:>10.2e}")
        else:
            print(f"{f:<40} NOT IN RESULTS — check column name")
else:
    print("No features selected yet. Run the notebook first and fill in SELECTED_FEATURES.")